# Beijing Air Tiantan Derivation

This provenance notebook records the decision log behind the `BeijingAir_Tiantan` benchmark dataset. It is derived from the Beijing Multi-Site Air Quality raw Tiantan station data and keeps an hourly local pollutant and weather series for forecasting robustness experiments.

The executable preprocessing contract lives in `scripts/preprocess_beijing_air_tiantan.py`. This notebook summarizes the rationale and checks behind the shipped benchmark dataset.

Original source: Chen, 2017, UCI Beijing Multi-Site Air Quality, https://doi.org/10.24432/C5RK5G
Benchmark derived dataset: `BeijingAir_Tiantan`


## EDA Summary

1. Inspect the raw PRSA station files and enumerate the 12 monitoring stations available in the shared 2013-2017 study window.
2. Apply one gap-aware screening policy to every station: split candidate periods on PM2.5 gaps longer than 24 hours and non-target feature gaps longer than 3 days.
3. Select the station and window with the longest retained forecastable hourly history under that shared policy.
4. Apply causal preprocessing within the selected segment: encode wind direction as an integer code, forward fill within the segment, trim unresolved leading rows if needed, and keep rows in timestamp order.
5. Save the canonical Parquet file and verify the row count, column contract, timestamp order, channel typing, and absence of missing modeled values.

No backfill is used anywhere in the exported series. Benchmark window defaults stay in `configs/dataset_windows.yaml`, not in this notebook.


## Slice Rationale

The all-station screen used the same export policy for every station. Under target gaps `>24h` plus feature gaps `>3d`, Tiantan is the continuity leader with 1001.5 kept days.

Important alternatives from the station screen:

| Station | Role in screen | Result |
| --- | --- | --- |
| Tiantan | Selected continuity leader | 1001.5 kept days, corr=0.9640 to the cross-station median PM2.5 series, RMSE24=88.14 |
| Guanyuan | Strongest representativeness candidate | corr=0.9846 and MAE=7.81 to the cross-station median PM2.5 series, but only 629.8 usable days |
| Wanliu | Former continuity candidate before feature-gap screening | falls to 678.5 usable days because long CO, NO2, and O3 outages break the window |

Tiantan is selected because it keeps the longest forecastable hourly segment once target and feature gaps are handled consistently across the station set.


## Window and Channel Contract

| Check | Benchmark contract |
| --- | --- |
| Raw Tiantan shape | 35,064 rows and 18 raw columns |
| Exported window | 2014-06-03 10:00:00 to 2017-02-28 23:00:00 |
| Exported rows | 24,038 hourly rows |
| Stored columns | `datetime` plus 12 modeled channels |
| Continuous channels | 11 pollutant and weather variables |
| Discrete channels | integer-coded wind direction `wd` |
| Target alias | `pm25` |
| Repository key | `BeijingAir_Tiantan` |
| Expected file | `data/processed/beijing_air_tiantan.parquet` |

Exact 24-hour PM2.5 gaps are allowed. PM2.5 gaps longer than 24 hours create target boundaries. Non-target feature outages longer than 3 days also create boundaries before filling. This trims the early Tiantan period that contains longer SO2, CO, NO2, and O3 outages.

All local pollutant, weather, and wind variables are kept. `wd` is stored as an integer-coded discrete channel. The other 11 modeled variables are continuous channels.


## Preprocessing Contract

- Build a complete hourly grid for the Tiantan station.
- Split first on PM2.5 gaps longer than 24 hours and modeled feature gaps longer than 3 days.
- Select the longest remaining segment under that policy.
- Forward fill only within the selected segment.
- Trim unresolved leading rows if needed after forward fill.
- Reject the export if any modeled value remains missing or non-finite.
- Write rows in strictly increasing hourly timestamp order.

## Benchmark Validation

Prepare from raw data with `uv run python scripts/preprocess_beijing_air_tiantan.py --raw-source <path-to-PRSA_Data_Tiantan_20130301-20170228.csv-or-zip> --output data/processed/beijing_air_tiantan.parquet`.

Validate the staged benchmark file with `uv run python scripts/preprocess_beijing_air_tiantan.py --output data/processed/beijing_air_tiantan.parquet --validate-existing`.

The validation path checks the registry contract, row count, column order, hourly datetime column, channel typing, and finite modeled values.
